# Serial Hybrid Model— Methanol Reactor Digital Twin

**Idea in one line:** physics solves the reactor; machine learning only learns the reaction **rate** used inside the ODE.

```text
Plant inputs u
      ↓
ODE along bed length z
   mass / heat / pressure  ← physics (unchanged)
   rate r(T, P, C)         ← learned ML kinetics
      ↓
Twin outputs (T at 0.2–1 m, P_out, Q, conversion)
```

| Role | Variables |
|------|-----------|
| **Inputs (4)** | Inlet T (°C), Inlet P (bar_g), Heating media T (°C), Methanol flow (kgmole/h) |
| **Outputs (8)** | Temp_0.2m … Temp_1m (°C), P_out (bar_g), Q (kW), Conversion (%) |

**Reaction:** CH₃OH → CO + 2H₂  
**Data:** `Methanol Reforming Data.xlsx` (same cleaning as the revised physics model)

**Training (two stages):**
1. For each train case, find a rate scale `η*` that matches Excel conversion.
2. Sample local states along the bed and train an MLP: `(T, P, C_CH3OH, C_CO) → log10(r)`.
3. Put that learned `r` back inside the ODE for the digital twin.


## 1. Imports

In [ ]:
from pathlib import Path #used to create and manage file 

import matplotlib.pyplot as plt #used to create plots
import numpy as np #used for numnerical operations
import pandas as pd #used for data manipulation and analysis
from scipy.integrate import solve_ivp #used to solve ODE (Ordinary dufferential equation)
from scipy.optimize import minimize_scalar #used to find the minimum of a scalar function
from sklearn.metrics import mean_absolute_error, r2_score #used to evaluate the performance of the model
from sklearn.model_selection import train_test_split #used to split the data into training and testing sets
from sklearn.neural_network import MLPRegressor #used to create a multi-layer perceptron (MLP) regressor
from sklearn.pipeline import Pipeline #used to create a pipeline of transformations
from sklearn.preprocessing import StandardScaler #used to standardize the features

DATA_FILE = Path("Methanol Reforming Data.xlsx")
# T_heat_C is a per-case plant input from Methanol Reforming Data.xlsx
MEASUREMENT_POSITIONS_M = np.array([0.2, 0.4, 0.6, 0.8, 1.0]) #positions of the temperature measurements
P_ATM_BAR = 1.01325  #atomspheric pressure

INPUT_COLS = ["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"] #input columns
OUTPUT_COLS = ["T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C",
    "P_out_bar_g", "Q_total_kW", "X_overall_pct",] #output columns
#we have taken the temp measuremnts seperately as we will using them for plotting
TEMP_COLS = ["T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C"]

print("Libraries loaded.")
print("Data file:", DATA_FILE.resolve())


## 2. Load and clean `Methanol Reforming Data.xlsx`

Same Excel cleaning as `revised_Physics Model_1`:
- sum Excel segment heat flows and flip sign (Excel stores heat into the reactor as negative)
- convert overall conversion (`Conversion at 1m (X-5)`) from a fraction to percent


In [ ]:
#function to load the excel file 
def load_data(path: Path) -> pd.DataFrame:
    # Load operating cases from Methanol Reforming Data.xlsx (Sheet1).
    raw = pd.read_excel(path, sheet_name=0, header=0, skiprows=[1])  # row1 = units
    n_raw = len(raw)

    data = raw.rename(
        columns={
            "Inlet Temperature (Ti)": "T_in_C",
            "Inlet Pressure (Pi)": "P_in_bar_g",
            "Heating media Temp (Th)": "T_heat_C",
            "Inlet molar flow rate (mi)": "F_in_kgmole_h",
            "Temperature at 0.2m (T1)": "T_0.2m_C",
            "Temperature at 0.4m (T2)": "T_0.4m_C",
            "Temperature at 0.6m (T3)": "T_0.6m_C",
            "Temperature at 0.8m (T4)": "T_0.8m_C",
            "Temperature at 1m (T5)": "T_1m_C",
            "Pressure at 1m (P5)": "P_out_bar_g",
        }
    )

    # Excel segment heat flows are negative for heat into the reactor → flip and sum.
    # Ensure numeric dtypes after dropping the Excel units row
    for _col in [
        "T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h",
        "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C", "P_out_bar_g",
        "Conversion at 1m (X-5)",
    ]:
        if _col in data.columns:
            data[_col] = pd.to_numeric(data[_col], errors="coerce")
    for _col in [c for c in data.columns if str(c).startswith("Heat flow")]:
        data[_col] = pd.to_numeric(data[_col], errors="coerce")

    heat_cols = [c for c in data.columns if str(c).startswith("Heat flow")]
    data["Q_total_kW"] = -data[heat_cols].sum(axis=1)

    # Excel stores conversion as a fraction → percent.
    data["X_overall_pct"] = data["Conversion at 1m (X-5)"] * 100.0

    print(f"Total rows in file          : {n_raw}")

    ok = (
        data["X_overall_pct"].between(0, 100)
        & data["Q_total_kW"].between(-50, 50)
        & data[["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"]].notna().all(axis=1)
    )
    cleaned = data.loc[ok].reset_index(drop=True)
    print(f"Valid rows                  : {len(cleaned)}")
    return cleaned
df = load_data(DATA_FILE)
display(df[INPUT_COLS + OUTPUT_COLS].head())


## 3. Fixed reactor geometry and physical constants

These stay in the ODE. Only the **rate law** will be replaced by ML.

Thermo (same as A3):
- **R** = `1.987` kcal/(kgmole·K)
- **ΔH(25 °C)** = `2.2e+04` kcal/kgmole
- **ΔH(T)** = ΔH(25°C) + ΔCp·(T − 25)  (temperature-dependent)


In [ ]:
# --- Thermo constants (same as A3) ---
R_kcal = 1.987                 # gas constant [kcal/(kgmole·K)]
delta_H_ref_kcal = 2.2e4       # reaction heat at 25 °C [kcal/kgmole]
T_ref_C = 25.0                 # reference temperature [°C]

# Convert to SI for the ODE: kcal/kgmole -> J/mol
KCAL_PER_KGMOLE_TO_J_PER_MOL = 4184.0 / 1000.0
R = R_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL                      # [J/(mol·K)]
delta_H_ref = delta_H_ref_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL  # [J/mol] at 25 °C

# Fixed reactor geometry parameters
D = 0.1            # Reactor diameter [m]
L = 1.0            # Reactor length [m]
h_o = 115.0     # Jacket-side HTC [W/(m²·C)]. The ODE uses h_heat = U(F_in).
# T_heat_C comes from each operating case (plant input)
epsilon = 0.33     # Reactor void fraction
dp = 1.0e-3        # Catalyst particle diameter [m]

# Cross-sectional area and wall perimeter of reactor [m²], [m]
Ac = np.pi * D**2 / 4    # Cross-sectional area [m²]
perimeter = np.pi * D    # Perimeter [m]

# Molecular weights [kg/mol]
MW_CH3OH = 32.04e-3   # Methanol
MW_CO    = 28.01e-3   # Carbon monoxide
MW_H2    = 2.016e-3   # Hydrogen

# Heat capacities of gases [J/(mol·K)]
Cp_CH3OH = 43.0   # Methanol
Cp_CO    = 29.0   # Carbon monoxide
Cp_H2    = 29.0   # Hydrogen

# ΔCp for reaction CH3OH -> CO + 2H2  [J/(mol·K)]
delta_Cp = Cp_CO + 2.0 * Cp_H2 - Cp_CH3OH

# Baseline LHHW coefficients (used only to create rate labels for training)
A_kinetic = 7.0e10    # Pre-exponential factor for main reaction rate
E_kinetic = 19120.0   # Activation energy [kcal/kgmole]; Arrhenius uses R_kcal
A_CO = 1.0e7          # Pre-exponential factor for CO adsorption
E_CO = 6036.8         # CO adsorption energy [kcal/kgmole]; Arrhenius uses R_kcal



mu_gas = 2.0e-5          # Gas viscosity [Pa·s]
k_gas = 0.035            # Gas thermal conductivity [W/(m·K)]


def bed_side_htc(F_in_mol_s):
    # Specchia bed-side wall coefficient h_i [W/(m²·°C)]
    mass_flux = F_in_mol_s * MW_CH3OH / Ac
    Re_p = dp * mass_flux / mu_gas
    Nu_w = 2.0 * epsilon + 0.0835 * (Re_p ** 0.91)
    return Nu_w * k_gas / dp


def overall_U(F_in_mol_s):
    # Overall wall coefficient. h_heat = U, from the inlet methanol flow.
    h_i = bed_side_htc(F_in_mol_s)
    return 1.0 / (1.0 / h_o + 1.0 / h_i)

def reaction_enthalpy(T_C):
    # Temperature-dependent ΔH [J/mol]: ΔH(T) = ΔH(25°C) + ΔCp*(T-25)
    return delta_H_ref + delta_Cp * (float(T_C) - T_ref_C)


print(f"R = {R_kcal} kcal/(kgmole·K)  ->  {R:.6f} J/(mol·K)")
print(f"ΔH(25°C) = {delta_H_ref_kcal:.3e} kcal/kgmole  ->  {delta_H_ref:.3e} J/mol")
print(f"ΔCp = {delta_Cp:.2f} J/(mol·K)")
print(f"ΔH(200°C) = {reaction_enthalpy(200)/4184*1000:.3e} kcal/kgmole")
print(f"D={D} m, L={L} m, T_heat=per-case input")


## 4. Unit converters + baseline LHHW rate (for labeling only)
we use LHHW to estimate the missing rate values so ML has something to learn from.It is working as a temporary teacher for ML.
LHHW rate means the reaction speed calculated with a Langmuir–Hinshelwood–Hougen–Watson formula.
It is a common catalytic kinetics model: reaction happens on the catalyst surface, and some species stick to (adsorb on) that surface.
Rate= how fast methanol turns into CO + H2

In [ ]:
#function to convert bar_g to pascal
def bar_g_to_Pa(P_bar_g):
    return (P_bar_g + P_ATM_BAR) * 1.0e5

#function to convert pascal to bar_g
def Pa_to_bar_g(P_Pa):
    return P_Pa / 1.0e5 - P_ATM_BAR

#function to convert kgmole/h to mol/s
def kgmole_h_to_mol_s(F_kgmole_h):
    return F_kgmole_h * 1000.0 / 3600.0


def concentrations(T_C, P, F_CH3OH, F_CO, F_H2):
    # Calculates ideal-gas concentrations [mol/m³] for CH3OH and CO from local flows.
    T_abs = T_C + 273.15  # Convert temperature from Celsius to Kelvin
    F_total = max(F_CH3OH + F_CO + F_H2, 1e-12)  # Total molar flow, avoid zero division
    C_total = P / (R * T_abs) / 1000.0  # kgmole/m³
    C_CH3OH = (F_CH3OH / F_total) * C_total  # CH3OH concentration
    C_CO = (F_CO / F_total) * C_total        # CO concentration
    return C_CH3OH, C_CO


def rate_lhhw(T_C, P, F_CH3OH, F_CO, F_H2, eta=1.0):
    """
    Calculates the baseline LHHW (Langmuir–Hinshelwood–Hougen–Watson) reaction rate for methanol decomposition.
    Parameters:
        T_C (float): Temperature in Celsius.
        P (float): Pressure in Pa.
        F_CH3OH (float): Molar flow rate of CH3OH [mol/s].
        F_CO (float): Molar flow rate of CO [mol/s].
        F_H2 (float): Molar flow rate of H2 [mol/s].
        eta (float, optional): Rate scaling factor (default 1.0).
    Returns:
        tuple: (rate [mol/(m³·s)], C_CH3OH [mol/m³], C_CO [mol/m³])
    """
    # Compute local(specific location)concentrations of CH3OH and CO [mol/m³]
    C_CH3OH, C_CO = concentrations(T_C, P, F_CH3OH, F_CO, F_H2)
    
    # Convert temperature to Kelvin
    T_abs = T_C + 273.15
    
    # Numerator of the LHHW rate expression (Arrhenius factor × CH3OH concentration)
    num = A_kinetic * np.exp(-E_kinetic / (R_kcal * T_abs)) * C_CH3OH

    # Denominator includes Langmuir-type adsorption term for CO
    den = 1.0 + A_CO * np.exp(-E_CO / (R_kcal * T_abs)) * C_CO

    # Final rate expression, scaled by eta
    return eta * (num / den) * 1000.0, C_CH3OH, C_CO  # rate in mol/(m³·s)


print("Baseline LHHW rate helper ready.")


This ## 5. Reactor ODE with a pluggable rate function
This section explains that the reactor's mass, energy, and pressure changes are always calculated using physical equations. The only part that can be swapped (or "plugged in") is the reaction rate function (rate_fn), which could be either a physics-based formula (LHHW) or a learned machine learning model.
Mass, energy, and Ergun pressure drop is calculated by physics.  
Only `rate_fn(...)` changes (baseline LHHW for labeling, then learned ML).

In [ ]:
def reactor_model(z, y, rate_fn, h_heat, T_heat_C):
    # State vector: [F_CH3OH, F_CO, F_H2, T_C, P_Pa, Q_W]
    F_CH3OH = max(y[0], 1e-12)   # Methanol flow rate [mol/s], bounded to avoid zeros
    F_CO    = max(y[1], 0.0)     # Carbon monoxide flow rate [mol/s], non-negative
    F_H2    = max(y[2], 0.0)     # Hydrogen flow rate [mol/s], non-negative
    T_C     = y[3]               # Temperature [Celsius]
    P       = max(y[4], 1.0e3)   # Pressure [Pa], bounded to avoid nonphysical values

    # Evaluate reaction rate (provided rate function, e.g., LHHW or ML surrogate)
    r = rate_fn(T_C, P, F_CH3OH, F_CO, F_H2)
    r = max(float(r), 0.0)       # Enforce non-negative rate for safety
    rpl = r * Ac                 # Rate per length (multiply by cross-sectional area)

    # Mass balances for plug flow (stoichiometry: CH3OH → CO + 2H2)
    dF_CH3OH_dz = -rpl           # Methanol consumed
    dF_CO_dz    = rpl            # CO produced
    dF_H2_dz    = 2.0 * rpl      # H2 produced

    # Heat transfer calculations
    q_transfer = h_heat * perimeter * (T_heat_C - T_C)  # Heat exchanged [W/m]
    # Total heat capacity flow in mixture [J/K/s], bounded to avoid division by zero
    Cp_flow = max(F_CH3OH * Cp_CH3OH + F_CO * Cp_CO + F_H2 * Cp_H2, 1e-9)
    # Energy balance with temperature-dependent ΔH(T) (same as A3)
    delta_H_T = reaction_enthalpy(T_C)  # [J/mol] at local temperature
    dT_dz = (q_transfer - delta_H_T * rpl) / Cp_flow

    # Pressure drop: Ergun equation for packed beds
    F_total  = F_CH3OH + F_CO + F_H2
    mass_flow = F_CH3OH * MW_CH3OH + F_CO * MW_CO + F_H2 * MW_H2  # [kg/s]
    MW_mix   = mass_flow / max(F_total, 1e-12)                    # Mixture MW [kg/mol]
    T_abs    = T_C + 273.15                                       # [K]
    rho      = P * MW_mix / (R * T_abs)                           # Mixture density [kg/m³]
    velocity = mass_flow / max(rho * Ac, 1e-12)                   # Superficial velocity [m/s]
    mu       = 2.0e-5                                             # Viscosity [Pa.s], assumed constant

    # Ergun equation terms
    viscous  = 150.0 * mu * (1.0 - epsilon) ** 2 / (epsilon ** 3 * dp ** 2) * velocity
    inertial = 1.75 * rho * (1.0 - epsilon) / (epsilon ** 3 * dp) * velocity ** 2
    dP_dz = -(viscous + inertial)                                 # Negative: pressure drops along z

    dQ_dz = q_transfer                                            # Total heat added as a tracked variable

    # Return derivatives as required by the ODE integrator
    return [dF_CH3OH_dz, dF_CO_dz, dF_H2_dz, dT_dz, dP_dz, dQ_dz]


def simulate(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, rate_fn, n_points=80):
    """
    Integrate the reactor ODE system from inlet (z=0) to outlet (z=L).
    Uses the provided rate function for reaction kinetics.
    Returns a dictionary of key outputs (temperatures, conversion, etc.)
    """
    # Convert and prepare inlet conditions
    F_in     = kgmole_h_to_mol_s(F_in_kgmole_h)         # [mol/s]
    P_in_Pa  = bar_g_to_Pa(P_in_bar_g)                  # [Pa]
    h_heat   = overall_U(F_in)                          # revised physics: h_heat = U(F_in)
    y0       = [F_in, 0.0, 0.0, float(T_in_C), P_in_Pa, 0.0]  # Initial state
    z_eval   = np.linspace(0.0, L, n_points)            # Points to evaluate along the reactor

    # Solve ODE system with scipy's solve_ivp
    sol = solve_ivp(
        lambda z, y: reactor_model(z, y, rate_fn, h_heat, float(T_heat_C)),  # ODE; h_heat = U(F_in)
        (0.0, L),                                      # Span from z=0 to z=L
        y0,                                            # Initial state vector
        t_eval=z_eval,                                 # Evaluation points
        method="BDF",                                  # Stiff solver (suitable for nonlinear systems)
        rtol=1e-5, atol=1e-8,                          # Integrator tolerances
    )
    if not sol.success:
        raise RuntimeError(sol.message)

    # Extract outputs at reactor outlet and at measurement positions
    F_out = float(sol.y[0, -1])                        # Methanol flow at outlet
    T_axial = np.interp(MEASUREMENT_POSITIONS_M, sol.t, sol.y[3])   # Interpolate T along bed
    X_pct = (F_in - F_out) / F_in * 100.0                             # Overall conversion in %

    # Build and return output dictionary
    return {
        "T_0.2m_C": float(T_axial[0]),        # Temperatures at specific axial positions [C]
        "T_0.4m_C": float(T_axial[1]),
        "T_0.6m_C": float(T_axial[2]),
        "T_0.8m_C": float(T_axial[3]),
        "T_1m_C":   float(T_axial[4]),
        "P_out_bar_g": float(Pa_to_bar_g(sol.y[4, -1])),   # Outlet pressure [barg]
        "Q_total_kW":  float(sol.y[5, -1] / 1000.0),       # Total heat added [kW]
        "X_overall_pct": float(X_pct),                     # Overall conversion [%]
        "z_m": sol.t,                                      # Positions array [m]
        "T_profile_C": sol.y[3],                           # Temperature profile along bed [C]
        "sol": sol,                                        # Full solution object (for advanced use)
        "F_in": F_in,                                      # Inlet flow rate [mol/s]
    }


def outputs_vector(pred):
    """
    Converts dictionary of simulation outputs
    into a numpy vector of the primary outputs of interest.
    """
    return np.array([pred[c] for c in OUTPUT_COLS], dtype=float)


print("Pluggable ODE simulator ready.")


## 6. Train / test split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Test rows : {len(test_df)}")

## 7. Stage 1 — create rate labels from Excel conversion
Q. What we are doing in this step?
A. We are generating supervised training data for reaction rate prediction by finding, for each training case, the scaling factor (η*) that makes the simulated conversion match the experimental (Excel) value. We then sample the reactor bed to get features and store local rates (η* × r_LHHW) as targets, since the local rate isn't directly provided by Excel.
Excel does not give the local rate `r`.  
So for each train case we find `η*` such that baseline LHHW × `η*` matches Excel conversion.  
Then we sample the bed and store local features → target rate `r* = η* · r_LHHW`.

In [ ]:
from typing import Any


def best_eta_for_row(row, n_points=40):
    """
    Finds eta* so simulated conversion matches experimental value.

    What is eta?
    -------------
    Eta (η) in this context is a multiplicative scaling factor applied to the kinetic rate law
    (often called an "activity factor" or "rate constant modifier"). It effectively tunes the
    overall rate predicted by the model to match the experimentally observed (Excel) conversion
    under specific conditions. We optimize eta* such that the simulated overall conversion
    (given the baseline LHHW kinetics) matches the value observed experimentally for that run.
    """
    # Experimental conversion
    X_excel = float(row["X_overall_pct"])

    def objective(log_eta):
        """
        Returns squared error between simulated and experimental conversion for this eta.
        """
        eta = float(10 ** log_eta)

        def rate_fn(T_C, P, F_M, F_CO, F_H2):
            # Rate law scaled by eta
            r, _, _ = rate_lhhw(T_C, P, F_M, F_CO, F_H2, eta=eta)
            return r

        try:
            # Simulate with current eta and get squared error
            pred = simulate(
                row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"],
                rate_fn, n_points=n_points,
            )
            return (pred["X_overall_pct"] - X_excel) ** 2
        except Exception:
            # Return large error if simulation fails
            return 1.0e6

    # Optimize log10(eta) to minimize error
    result = minimize_scalar(
        objective, bounds=(-2.0, 2.0), method="bounded", options={"xatol": 0.08}
    )
    return float(10 ** result.x)


def build_rate_training_set(data, n_cases=250, sample_stride=4, random_state=42):
    """
    For each random training case, find eta*, simulate, and label sampled states.
    Returns arrays for features and labels.
    """
    # Sample n_cases for rate fitting
    sample = data.sample(n=min(n_cases, len(data)), random_state=random_state)
    X_feat = []         # Features per bed location
    y_log10_r = []      # Target log10(r) values
    etas = []           # Stored eta* values

    print(f"Building rate labels from {len(sample)} train cases...")
    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        # Find eta* for operating point
        eta_star = best_eta_for_row(row)
        etas.append(eta_star)

        def rate_fn(T_C, P, F_M, F_CO, F_H2, eta=eta_star):
            # Scaled rate law, fixed eta*
            r, _, _ = rate_lhhw(T_C, P, F_M, F_CO, F_H2, eta=eta)
            return r

        # Simulate bed profile with this rate law
        pred = simulate(
            row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"],
            rate_fn, n_points=40,
        )
        sol = pred["sol"]

        # Sample bed to get features and labels
        # For each sampled reactor state along the bed, compute features and target: (T, P, concentrations) and log10(rate)
        for j in range(0, sol.y.shape[1], sample_stride):
            F_M, F_CO, F_H2 = sol.y[0, j], sol.y[1, j], sol.y[2, j]
            T_C, P = sol.y[3, j], sol.y[4, j]
            r_star, C_M, C_CO = rate_lhhw(T_C, P, F_M, F_CO, F_H2, eta=eta_star)
            X_feat.append([T_C, P / 1.0e5, C_M, C_CO])      # Store features
            y_log10_r.append(np.log10(max(r_star, 1e-30)))  # Store clipped log10(r)
    

        # Show progress
        if i % 50 == 0 or i == len(sample):
            print(f"  labeled {i}/{len(sample)} cases")

    return np.asarray(X_feat), np.asarray(y_log10_r), np.asarray(etas)


# Generate rate model training data
X_rate, y_log10_r, eta_labels = build_rate_training_set(train_df, n_cases=250)
print(f"Rate samples: {len(X_rate)}")
print(f"η* median = {np.median(eta_labels):.4e}  (min={eta_labels.min():.4e}, max={eta_labels.max():.4e})")


## 8. Stage 2 — train ML kinetics model
This step trains a machine learning model (MLPRegressor) to predict the reaction rate logarithm (log10(r)) as a function of local reactor state variables (temperature, pressure, concentrations). The model learns from simulated data, and its predictions are later exponentiated (10**prediction) to give the actual rate r for use inside the reactor ODE simulation. This lets the physics-based simulation use a learned, data-driven kinetics law instead of hand-coded equations. Methanol depletion is handled by forcing r = 0 if its concentration is too low.
MLP maps local state → `log10(r)`.  
Inside the ODE we convert back to `r = 10 ** prediction` and force `r = 0` if methanol is gone.

In [ ]:
# Build pipeline: standardize inputs, then fit MLP to predict log10(rate)
kinetics_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPRegressor(
                hidden_layer_sizes=(32, 16),  # Two layers with 32 and 16 units
                activation="tanh",            # Nonlinear activation
                solver="adam",                # Optimizer
                max_iter=1000,                # Max training steps
                random_state=42,              # Seed for reproducibility
            ),
        ),
    ]
)
kinetics_model.fit(X_rate, y_log10_r)  # Train on simulated data
print("Learned kinetics MLP trained.")


def rate_learned(T_C, P, F_CH3OH, F_CO, F_H2):
    # Predict reaction rate using ML model
    C_CH3OH, C_CO = concentrations(T_C, P, F_CH3OH, F_CO, F_H2)
    if C_CH3OH < 1e-12:    # No methanol left
        return 0.0
    feat = np.array([[T_C, P / 1.0e5, C_CH3OH, C_CO]], dtype=float)  # Prepare input features
    log10_r = float(kinetics_model.predict(feat)[0])                  # Predict log10(rate)
    return max(10.0 ** log10_r, 0.0)                                 # Convert to rate, ensure non-negative


# Simple test at one input state
r_demo = rate_learned(200.0, bar_g_to_Pa(0.4), kgmole_h_to_mol_s(0.1), 0.0, 0.0)
print(f"Example learned rate at inlet-like state: {r_demo:.4e} mol/(m³·s)")

## 9. Digital twin function (learned kinetics inside ODE)

This step defines and documents the "digital twin" function, which uses the learned machine learning (ML) kinetics model inside a physics-based ODE reactor simulation. The function takes plant input conditions (temperature, pressure, flow) and predicts reactor outputs by simulating the reactor with the ML-predicted reaction rates, enabling data-driven process modeling.


In [ ]:
def predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h):
    # Takes 4 plant input values and predicts 8 plant outputs using ML inside physics simulation
    pred = simulate(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, rate_learned, n_points=80)  # Run simulation with learned kinetics
    out = {c: pred[c] for c in OUTPUT_COLS}  # Collect main outputs
    out["z_m"] = pred["z_m"]                # Add reactor positions
    out["T_profile_C"] = pred["T_profile_C"]  # Add reactor temperature profile
    return out

# Run prediction for the first data point as a basic test
demo = df.iloc[0]
demo_pred = predict_outputs(demo["T_in_C"], demo["P_in_bar_g"], demo["T_heat_C"], demo["F_in_kgmole_h"])

# Compare model prediction to Excel (original) values
smoke = pd.DataFrame({
    "Output": OUTPUT_COLS,
    "Excel": [demo[c] for c in OUTPUT_COLS],  # Reference outputs
    "Learned-kinetics twin": [demo_pred[c] for c in OUTPUT_COLS],  # Model outputs
})
smoke["Abs error"] = (smoke["Learned-kinetics twin"] - smoke["Excel"]).abs()  # Compute absolute errors
print("Smoke test — first cleaned Methanol Reforming Data.xlsx row:")
display(smoke.round(4))  # Show comparison


## 10. Validate on the test set

Compare:
- **Baseline physics:** one global median `η` × LHHW (no local ML)
- **Learned kinetics:** ML rate inside the ODE

In [ ]:
# Use median of eta values for baseline
ETA_GLOBAL = float(np.median(eta_labels))
print(f"Baseline global η (median of labels) = {ETA_GLOBAL:.6e}")


# Baseline rate function (LHHW with fixed eta)
def rate_baseline(T_C, P, F_M, F_CO, F_H2, eta=ETA_GLOBAL):
    r, _, _ = rate_lhhw(T_C, P, F_M, F_CO, F_H2, eta=eta)
    return r


Y_excel = []  # True output values from Excel/test set
Y_base = []   # Outputs using baseline eta model
Y_learn = []  # Outputs using learned kinetics model

print(f"Evaluating {len(test_df)} test cases...")
for i, row in test_df.iterrows():
    # Get true outputs for this row
    y_ex = np.array([row[c] for c in OUTPUT_COLS], dtype=float)
    try:
        # Simulate with baseline model
        y_b = outputs_vector(
            simulate(row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"], rate_baseline, n_points=50)
        )
        # Simulate with ML/learned rate model
        y_l = outputs_vector(
            simulate(row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"], rate_learned, n_points=50)
        )
    except Exception:
        continue  # Skip on failures
    Y_excel.append(y_ex)
    Y_base.append(y_b)
    Y_learn.append(y_l)
    if len(Y_excel) % 50 == 0:
        print(f"  done {len(Y_excel)}")

# Convert results to numpy arrays for analysis
Y_excel = np.vstack(Y_excel)
Y_base = np.vstack(Y_base)
Y_learn = np.vstack(Y_learn)
print(f"Scored {len(Y_excel)} test rows")

# Compute metrics by output variable (column)
rows = []
for i, col in enumerate(OUTPUT_COLS):
    rows.append({
        "Output": col,
        "MAE baseline η": mean_absolute_error(Y_excel[:, i], Y_base[:, i]),
        "MAE learned kinetics": mean_absolute_error(Y_excel[:, i], Y_learn[:, i]),
        "R2 learned": r2_score(Y_excel[:, i], Y_learn[:, i]),
    })
metrics = pd.DataFrame(rows)
display(metrics.round(4))  # Show errors/metrics

# Plot: Conversion and outlet T for both models vs. Excel
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Plot: Overall conversion
axes[0].scatter(Y_excel[:, 7], Y_base[:, 7], alpha=0.45, s=18, label="Baseline η")
axes[0].scatter(Y_excel[:, 7], Y_learn[:, 7], alpha=0.45, s=18, label="Learned kinetics")
lim = max(Y_excel[:, 7].max(), Y_learn[:, 7].max(), 1e-6) * 1.1
axes[0].plot([0, lim], [0, lim], "k--", lw=1)
axes[0].set_xlabel("Excel conversion (%)")
axes[0].set_ylabel("Model conversion (%)")
axes[0].set_title("Overall conversion")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot: 1m (outlet) temperature
axes[1].scatter(Y_excel[:, 4], Y_base[:, 4], alpha=0.45, s=18, label="Baseline η")
axes[1].scatter(Y_excel[:, 4], Y_learn[:, 4], alpha=0.45, s=18, label="Learned kinetics")
t0 = min(Y_excel[:, 4].min(), Y_learn[:, 4].min()) - 5
t1 = max(Y_excel[:, 4].max(), Y_learn[:, 4].max()) + 5
axes[1].plot([t0, t1], [t0, t1], "k--", lw=1)
axes[1].set_xlabel("Excel Temp_1m (°C)")
axes[1].set_ylabel("Model Temp_1m (°C)")
axes[1].set_title("Outlet temperature")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Learned kinetics inside ODE vs Excel (test set)")
fig.tight_layout()
plt.show()

# Note about what these results mean
print(
    "Note: at Excel conversions (~0.1–0.5%), reaction heat is small, "
    "so axial T is still dominated by wall heating. Learned kinetics "
    "mainly improves conversion (and flows); T/Q need heat-transfer "
    "calibration or a residual layer if you need Excel-level T accuracy."
)

---

## 11. Interactive digital twin

Run this cell and enter the **4 plant inputs**.  
The twin integrates the ODE with the **learned rate** inside.  
Type `q` to quit.

In [ ]:
def _ask_float(prompt, default=None):
    hint = f" [{default}]" if default is not None else ""
    while True:
        raw = input(f"{prompt}{hint}: ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            return None
        if raw == "" and default is not None:
            return float(default)
        try:
            return float(raw)
        except ValueError:
            print("  Please enter a number (or q to quit).")


def run_digital_twin_interactive():
    print("\n" + "=" * 62)
    print("  METHANOL REACTOR — LEARNED KINETICS DIGITAL TWIN")
    print("  ODE balances = physics | rate r = ML(T, P, C)")
    print("=" * 62)
    print("  Press Enter to accept the default in [brackets].")
    print("  Type q at any prompt to stop.\n")

    while True:
        T_in_C = _ask_float("Inlet temperature [°C]", default=float(df["T_in_C"].iloc[0]))
        if T_in_C is None:
            print("Digital twin stopped.")
            break

        P_in_bar_g = _ask_float("Inlet pressure [bar_g]", default=float(df["P_in_bar_g"].iloc[0]))
        T_heat_C = _ask_float(
            "Heating media temperature [C]",
            default=float(df["T_heat_C"].iloc[0]),  # Excel first-case default
        )
        if P_in_bar_g is None:
            print("Digital twin stopped.")
            break

        F_in_kgmole_h = _ask_float("Methanol molar flow [kgmole/h]", default=float(df["F_in_kgmole_h"].iloc[0]))
        if F_in_kgmole_h is None:
            print("Digital twin stopped.")
            break

        if F_in_kgmole_h <= 0:
            print("  Flow must be > 0. Try again.\n")
            continue

        try:
            pred = predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h)
        except Exception as exc:
            print(f"  Prediction failed: {exc}\n")
            continue

        print("\n" + "-" * 62)
        print("INPUTS")
        print(f"  Inlet Temperature     : {T_in_C:.2f} °C")
        print(f"  Inlet Pressure        : {P_in_bar_g:.4f} bar_g")
        print(f"  Methanol flow         : {F_in_kgmole_h:.4f} kgmole/h")
        print("\nTWIN OUTPUTS (learned kinetics inside ODE)")
        print(f"  Temp_0.2m             : {pred['T_0.2m_C']:.2f} °C")
        print(f"  Temp_0.4m             : {pred['T_0.4m_C']:.2f} °C")
        print(f"  Temp_0.6m             : {pred['T_0.6m_C']:.2f} °C")
        print(f"  Temp_0.8m             : {pred['T_0.8m_C']:.2f} °C")
        print(f"  Temp_1m (outlet)      : {pred['T_1m_C']:.2f} °C")
        print(f"  Outlet Pressure       : {pred['P_out_bar_g']:.4f} bar_g")
        print(f"  Total heat input      : {pred['Q_total_kW']:.4f} kW")
        print(f"  Overall Conversion    : {pred['X_overall_pct']:.4f} %")
        print("-" * 62)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(pred["z_m"], pred["T_profile_C"], lw=2, label="Twin T(z)")
        ax.scatter(
            MEASUREMENT_POSITIONS_M,
            [pred[c] for c in TEMP_COLS],
            color="crimson", zorder=5, label="Sampled T",
        )
        ax.axhline(T_heat_C, color="gray", ls="--", alpha=0.6, label=f"Heat medium ({T_heat_C:.0f} °C)")
        ax.set_xlabel("Reactor length (m)")
        ax.set_ylabel("Temperature (°C)")
        ax.set_title("Learned-kinetics digital twin — axial temperature")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        again = input("\nRun another case? [Y/n]: ").strip().lower()
        if again in {"n", "no", "q", "quit"}:
            print("Digital twin stopped.")
            break


run_digital_twin_interactive()